# Dataset statistics (Figs. S1-S5) and docking-score ranking power (Fig. S15)

Run from this directory (`figures/dataset/`). Inputs:

- `../inputs/final_dataset_..._resplit_NEW.csv`: the curated dataset table (Fig. S2).
- `data_curation/cached_data/metal_ligand_data_..._resplit.json`: metals per system
  (Fig. S3), written by `data_curation/generate_metal_ligand_data.py`.
- `chelation_cache.pkl` (Fig. S4) and `metal_ligand_distances_cache.pkl` (Fig. S5):
  per-system chelating residues and metal-ligand distances. The distance cache is too
  large for git (32 MB); see `figures/README.md`.
- `spearman_cache.pkl` (Fig. S15): built from the docking feature CSVs by
  `build_spearman_cache.py`.

Fig. S1 uses the filtering counts hard-coded in its cell. Plots go to `plots/`.


In [1]:
import json
import os
import pickle
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

SCRIPT_DIR = os.path.dirname(os.path.abspath("__file__"))
REPO = os.path.abspath(os.path.join(SCRIPT_DIR, "..", ".."))

JSON_PATH = os.path.join(
    REPO,
    "data_curation",
    "cached_data",
    "metal_ligand_data_distTresh=4.0_mwThresh=750.0_goodApoThresh=70.0_resplit.json",
)
CSV_PATH = os.path.join(
    REPO,
    "figures",
    "inputs",
    "final_dataset_distTresh=4.0_mwThresh=750.0_goodApoThresh=70.0_resplit_NEW.csv",
)
CACHE_PATH = os.path.join(SCRIPT_DIR, "chelation_cache.pkl")
DISTANCES_CACHE_PATH = os.path.join(SCRIPT_DIR, "metal_ligand_distances_cache.pkl")

SPLITS = ["train", "val", "test"]
SPLIT_COLORS = {"train": "#ef476f", "val": "#06d6a0", "test": "#118ab2"}
AMINO_ACIDS = {
    "ALA",
    "ARG",
    "ASN",
    "ASP",
    "CYS",
    "GLN",
    "GLU",
    "GLY",
    "HIS",
    "ILE",
    "LEU",
    "LYS",
    "MET",
    "PHE",
    "PRO",
    "SER",
    "THR",
    "TRP",
    "TYR",
    "VAL",
    "MSE",
    "SEC",
}

# ---- Font / figure sizes ----
fontsize_x_label = 8
fontsize_y_label = 8
fontsize_x_ticks = 8
fontsize_y_ticks = 8
fontsize_legend = 8
fontsize_titles = 8
cm = 1 / 2.54


def apply_axis_fonts(ax):
    ax.tick_params(axis="x", labelsize=fontsize_x_ticks)
    ax.tick_params(axis="y", labelsize=fontsize_y_ticks)


PLOT_DIR = os.path.join(os.path.dirname(os.path.abspath("__file__")), "plots")
os.makedirs(PLOT_DIR, exist_ok=True)

## Load data

In [2]:
df = pd.read_csv(CSV_PATH)
df = df[df["split"].isin(SPLITS)].copy()
print(df["split"].value_counts().to_dict())

with open(JSON_PATH) as f:
    metal_ligand_data = json.load(f)

with open(CACHE_PATH, "rb") as f:
    chelation_cache = pickle.load(f)
print(
    f"Chelation cache: {len(chelation_cache)} systems, "
    f"{sum(1 for v in chelation_cache.values() if v)} with interactions"
)

{'train': 40784, 'val': 1073, 'test': 156}


Chelation cache: 40044 systems, 34891 with interactions


## Metal identity distribution

In [3]:
records = []
for _, row in df.iterrows():
    entry = metal_ligand_data.get(row["system_id"], {})
    for metal_name, _ in entry.get("metals", []):
        records.append({"split": row["split"], "metal": metal_name})

metal_df = pd.DataFrame(records)
metal_df_train_val_test = metal_df[metal_df["split"].isin(SPLITS)]
metal_counts = metal_df_train_val_test["metal"].value_counts()
top_metals = metal_counts[metal_counts >= 100].index.tolist()
metal_df = metal_df[metal_df["metal"].isin(top_metals)]

pivot = metal_df.groupby(["split", "metal"]).size().reset_index(name="count")
pivot["fraction"] = pivot["count"] / pivot.groupby("split")["count"].transform("sum")

idx = pd.MultiIndex.from_product([SPLITS, top_metals], names=["split", "metal"])
pivot = pivot.set_index(["split", "metal"]).reindex(idx, fill_value=0).reset_index()

fig, ax = plt.subplots(figsize=(16 * cm, 8 * cm))
width = 0.25
x = np.arange(len(top_metals))
for i, split in enumerate(SPLITS):
    sub = (
        pivot[pivot["split"] == split]
        .set_index("metal")
        .reindex(top_metals, fill_value=0)
    )
    label = split.capitalize()
    if label == "Val":
        label = "Validation"
    ax.bar(
        x + i * width,
        sub["fraction"] * 100,
        width,
        label=label,
        color=SPLIT_COLORS[split],
        edgecolor="white",
        linewidth=0.5,
    )
ax.set_xticks(x + width)
ax.set_xticklabels(top_metals)
ax.set_ylabel("Occurrence (%)", fontsize=fontsize_y_label)
ax.legend(fontsize=fontsize_legend, frameon=False)
apply_axis_fonts(ax)
fig.tight_layout()
path = os.path.join(PLOT_DIR, "metal_occurrence_by_split.pdf")
fig.savefig(path, bbox_inches="tight")
plt.show()

## Molecular weight distribution

In [4]:
fig, ax = plt.subplots(figsize=(16 * cm, 8 * cm))
bins = np.linspace(
    df["ligand_molecular_weight"].min(), df["ligand_molecular_weight"].max(), 40
)
for split in SPLITS:
    sub = df[df["split"] == split]["ligand_molecular_weight"].dropna()
    label = split.capitalize()
    if label == "Val":
        label = "Validation"
    n = len(sub)
    n_str = f"{n:,}"
    ax.hist(
        sub,
        bins=bins,
        histtype="step",
        linewidth=1.8,
        color=SPLIT_COLORS[split],
        label=f"{label} (n={n_str})",
        density=True,
    )
ax.set_xlabel("Ligand Molecular Weight (Da)", fontsize=fontsize_x_label)
ax.set_ylabel("Density", fontsize=fontsize_y_label)
ax.set_yticks([])
ax.legend(fontsize=fontsize_legend, loc="upper left", frameon=False)
apply_axis_fonts(ax)
fig.tight_layout()
path = os.path.join(PLOT_DIR, "ligand_molecular_weight_distribution.pdf")
fig.savefig(path, bbox_inches="tight")
plt.show()

## Chelating amino acid frequency

Protein residues within 3.5 Å of a metal, considering only O/N/S donor atoms.

In [5]:
sid_to_split = df.set_index("system_id")["split"].to_dict()
records = []
for sid, entries in chelation_cache.items():
    split = sid_to_split.get(sid)
    if split not in SPLITS:
        continue
    for metal, res_name, dist in entries:
        if res_name in AMINO_ACIDS:
            res_name_formatted = res_name.capitalize()
            records.append(
                {
                    "split": split,
                    "metal": metal,
                    "residue": res_name_formatted,
                    "distance": dist,
                }
            )

chel_df = pd.DataFrame(records)
top_n = 10
counts = chel_df["residue"].value_counts().head(top_n)
colors = plt.cm.tab10.colors[:top_n]

fig, ax = plt.subplots(figsize=(16 * cm, 8 * cm))
ax.barh(counts.index[::-1], counts.values[::-1], color=colors[::-1], edgecolor="white")
ax.set_xlabel(
    "Number of Metal–Residue Interactions",
    fontsize=fontsize_x_label,
)
apply_axis_fonts(ax)
fig.tight_layout()
path = os.path.join(PLOT_DIR, "top_chelating_amino_acids.pdf")
fig.savefig(path, bbox_inches="tight")
plt.show()

## Metal-ligand distances

Distribution of distances from ligand heteroatoms (N, O, S) to metal centres (≤ 4 Å).

In [6]:
# CSD DISTANCES
# metal-heteroatom distances observed in the Cambridge Crystallographic Database
# for O, we consider the distance to water oxygens
# Reference: https://pubs.acs.org/doi/pdf/10.1021/jp807972e?ref=article_openPDF Table 4

CSD_dist_MG_O = 2.06
CSD_dist_MG_S = None  # (not enough data for Mg-S in CSD)
CSD_dist_MG_N = 2.19

CSD_dist_MN_O = 2.18
CSD_dist_MN_S = 2.60
CSD_dist_MN_N = 2.27

CSD_dist_ZN_O = 1.99
CSD_dist_ZN_S = 2.32
CSD_dist_ZN_N = 2.03


CSD_REF = {
    "Mg": {"O": CSD_dist_MG_O, "N": CSD_dist_MG_N, "S": CSD_dist_MG_S},
    "Mn": {"O": CSD_dist_MN_O, "N": CSD_dist_MN_N, "S": CSD_dist_MN_S},
    "Zn": {"O": CSD_dist_ZN_O, "N": CSD_dist_ZN_N, "S": CSD_dist_ZN_S},
}

In [7]:
with open(DISTANCES_CACHE_PATH, "rb") as f:
    dist_df = pickle.load(f)
print(f"Metal-ligand distances: {len(dist_df):,} atom-metal pairs")
dist_df["element"] = dist_df["element"].str.upper()

MAX_DIST = 4.0
PALETTE_NSO = {"N": "#8FA8FF", "O": "#FF9E9E", "S": "#F7E58C"}
subset = dist_df[dist_df["distance"] <= MAX_DIST].copy()


def _hist_ax(
    ax, data_series_dict, palette, title, max_dist=4.0, bins=40, ref_lines=None
):
    has_data = False
    for label, series in data_series_dict.items():
        series = series.dropna()
        if len(series) < 5:
            continue
        ax.hist(
            series,
            bins=bins,
            range=(0, max_dist),
            density=True,
            histtype="step",
            linewidth=1.8,
            color=palette.get(label, "#333333"),
            label=f"{label} (n={len(series):,})",
        )
        has_data = True

        if ref_lines and label in ref_lines and ref_lines[label] is not None:
            ax.axvline(
                ref_lines[label],
                color=palette.get(label, "#333333"),
                linestyle="--",
                linewidth=2.0,
                alpha=0.9,
            )

    ax.set_xlim(1, max_dist)
    ax.set_xlabel("Distance to metal (Å)", fontsize=fontsize_x_label)
    ax.set_yticks([])
    ax.set_ylabel("Density", fontsize=fontsize_y_label)
    ax.set_title(title, fontsize=fontsize_titles)
    apply_axis_fonts(ax)

    if has_data:
        ax.legend(fontsize=fontsize_legend, framealpha=0.7, loc="upper left")
    else:
        ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)


fig, axes = plt.subplots(2, 2, figsize=(16 * cm, 16 * cm), constrained_layout=True)

nso_all = {
    e: subset.loc[subset["element"] == e, "distance"]
    for e in ["N", "O", "S"]
    if (subset["element"] == e).any()
}
_hist_ax(axes[0, 0], nso_all, PALETTE_NSO, "All metals")

for ax, metal in zip([axes[0, 1], axes[1, 0], axes[1, 1]], ["Mg", "Mn", "Zn"]):
    msub = subset[subset["metal_symbol"] == metal]
    nso = {
        e: msub.loc[msub["element"] == e, "distance"]
        for e in ["N", "O", "S"]
        if (msub["element"] == e).any()
    }

    _hist_ax(ax, nso, PALETTE_NSO, metal, ref_lines=CSD_REF.get(metal, {}))

path = os.path.join(PLOT_DIR, "metal_ligand_distance_distributions.pdf")
fig.savefig(path, bbox_inches="tight")
plt.show()

Metal-ligand distances: 772,790 atom-metal pairs


## Docking score – RMSD Spearman correlation

Distribution of per-system Spearman ρ between docking score and RMSD metrics, stratified by train / val / test split.

In [8]:
from matplotlib.lines import Line2D
from scipy.stats import gaussian_kde

with open(os.path.join(SCRIPT_DIR, "spearman_cache.pkl"), "rb") as f:
    spearman_cache = pickle.load(f)

DOCKING_METHODS = ("VINA", "SMINA", "GNINA")
DOCKING_RMSD_COLS = {
    "MBG-RMSD": "close_rmsd_3.5_hetero",
    "Molecule RMSD": "symm_rmsd",
}
METHOD_DISPLAY = {"VINA": "Vina", "SMINA": "smina", "GNINA": "GNINA"}
COL_SUPERTITLES = {
    "MBG-RMSD": "MBG-RMSD",
    "Molecule RMSD": "Whole-molecule RMSD",
}

n_rows = len(DOCKING_METHODS)
n_cols = len(DOCKING_RMSD_COLS)

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(16 * cm, 24 * cm),
)

for row_i, method in enumerate(DOCKING_METHODS):
    for col_i, (rmsd_label, rmsd_col) in enumerate(DOCKING_RMSD_COLS.items()):
        ax = axes[row_i, col_i]
        method_data = spearman_cache.get(method, {}).get(rmsd_col, {})
        split_ns = {}
        for split in SPLITS:
            rhos = method_data.get(split, np.array([]))
            if len(rhos) < 3:
                continue
            split_ns[split] = len(rhos)
            color = SPLIT_COLORS[split]
            kde = gaussian_kde(rhos, bw_method="scott")
            x_grid = np.linspace(-1.0, 1.0, 300)
            ax.plot(x_grid, kde(x_grid), color=color, linewidth=1.6)
            ax.axvline(np.mean(rhos), color=color, linestyle=":", linewidth=1.4)
            print(
                f"{method} {rmsd_label} {split}: n={len(rhos):,}, mean={np.mean(rhos):.3f}, std={np.std(rhos):.3f}"
            )
        ax.axvline(0, color="lightgrey", linestyle="-", linewidth=0.6, zorder=0)
        ax.set_xlim(-1, 1)
        ax.set_ylim(bottom=0)
        ax.set_xlabel("Spearman ρ", fontsize=fontsize_x_label)
        ax.set_ylabel("Density", fontsize=fontsize_y_label)
        ax.set_yticks([])
        if col_i == 0:
            # include a, b, c
            ax.set_title(
                f"{chr(97 + row_i)}) {METHOD_DISPLAY[method]}",
                fontsize=fontsize_titles,
                loc="left",
            )
        apply_axis_fonts(ax)
        n_lines = "\n".join(
            f"{'Validation' if s == 'val' else s.capitalize()}: n={n:,}"
            for s, n in split_ns.items()
        )
        ax.text(
            0.97,
            0.97,
            n_lines,
            transform=ax.transAxes,
            fontsize=fontsize_legend,
            color="black",
            va="top",
            ha="right",
            linespacing=1.5,
        )

fig.tight_layout()
fig.subplots_adjust(top=0.88, bottom=0.20, hspace=0.3)

for col_i, rmsd_label in enumerate(DOCKING_RMSD_COLS.keys()):
    ax_top = axes[0, col_i]
    pos = ax_top.get_position()
    x_center = pos.x0 + pos.width / 2
    fig.text(
        x_center,
        0.9,
        COL_SUPERTITLES[rmsd_label],
        ha="center",
        va="bottom",
        fontsize=fontsize_titles,
    )

handles_dataset = [
    Line2D([0], [0], color=SPLIT_COLORS["train"], linewidth=1.6),
    Line2D([0], [0], color=SPLIT_COLORS["val"], linewidth=1.6),
    Line2D([0], [0], color=SPLIT_COLORS["test"], linewidth=1.6),
]
labels_dataset = ["Train", "Validation", "Test"]

handles_style = [
    Line2D([0], [0], color="grey", linewidth=1.6, linestyle="-"),
    Line2D([0], [0], color="grey", linewidth=1.6, linestyle=":"),
]
labels_style = ["KDE", "Mean correlation"]

leg1 = fig.legend(
    handles_dataset,
    labels_dataset,
    title="Dataset",
    loc="lower center",
    bbox_to_anchor=(0.3, 0.08),
    fontsize=fontsize_legend,
    title_fontsize=fontsize_legend,
    frameon=False,
    ncol=1,
)
fig.legend(
    handles_style,
    labels_style,
    loc="lower center",
    bbox_to_anchor=(0.72, 0.08),
    fontsize=fontsize_legend,
    frameon=False,
    ncol=1,
)
fig.add_artist(leg1)

path = os.path.join(PLOT_DIR, "spearman_correlation_distributions.pdf")
fig.savefig(path, bbox_inches="tight")
plt.show()

VINA MBG-RMSD train: n=3,115, mean=-0.245, std=0.246
VINA MBG-RMSD val: n=880, mean=-0.246, std=0.226
VINA MBG-RMSD test: n=118, mean=-0.250, std=0.188
VINA Molecule RMSD train: n=3,115, mean=-0.281, std=0.274
VINA Molecule RMSD val: n=880, mean=-0.299, std=0.236
VINA Molecule RMSD test: n=118, mean=-0.287, std=0.181
SMINA MBG-RMSD train: n=3,081, mean=-0.252, std=0.252
SMINA MBG-RMSD val: n=859, mean=-0.241, std=0.233
SMINA MBG-RMSD test: n=118, mean=-0.247, std=0.189
SMINA Molecule RMSD train: n=3,081, mean=-0.291, std=0.279
SMINA Molecule RMSD val: n=859, mean=-0.300, std=0.240
SMINA Molecule RMSD test: n=118, mean=-0.287, std=0.187
GNINA MBG-RMSD train: n=3,367, mean=-0.349, std=0.279
GNINA MBG-RMSD val: n=901, mean=-0.389, std=0.260
GNINA MBG-RMSD test: n=118, mean=-0.321, std=0.245
GNINA Molecule RMSD train: n=3,367, mean=-0.450, std=0.315
GNINA Molecule RMSD val: n=901, mean=-0.518, std=0.286
GNINA Molecule RMSD test: n=118, mean=-0.394, std=0.259


## Dataset filtering steps

In [9]:
train_counts = [418381, 413683, 379831, 307305, 38292, 34872, 40784]
removed_counts = [146420, 144652, 133560, 100418, 18807, 16610, 9555]
test_counts = [1436, 1420, 1399, 1089, 178, 160, 156]
val_counts = [1157, 1157, 1099, 839, 154, 144, 1073]
steps = [
    "All",
    "No steric clashes",
    "Ligand MW < 750 Da",
    "Proper ligand",
    "Metalloproteins",
    "M-L distance < 4 Å",
    "Resplit",
]

In [10]:
fig, ax = plt.subplots(figsize=(16 * cm, 16 * cm))

w = 0.8 / 2 / 2
d = w / 2

x = np.arange(len(steps))

ax.bar(x - w - d, removed_counts, color="#D3D3D3", label="Removed", width=w)
ax.bar(x - d, train_counts, color=SPLIT_COLORS["train"], label="Train", width=w)
ax.bar(x + d, val_counts, color=SPLIT_COLORS["val"], label="Validation", width=w)
ax.bar(x + d + w, test_counts, color=SPLIT_COLORS["test"], label="Test", width=w)

ax.set_yscale("log")
ax.set_yticks(
    np.logspace(0, 6, num=7),
    labels=["1", "10", "100", "1k", "10k", "100k", "1M"],
)
ax.set_ylim(1, 1e7)
ax.set_ylabel("Number of systems", fontsize=fontsize_y_label)
ax.set_xticks(x)
ax.set_xticklabels(steps, rotation=45, ha="center", fontsize=fontsize_x_ticks)

log_spacer = lambda y_value: 10 ** (np.log10(y_value) + 0.1)

for i in range(len(steps)):
    ax.text(
        i - w - d,
        log_spacer(removed_counts[i]),
        f"{removed_counts[i]:,}",
        ha="center",
        va="bottom",
        fontsize=fontsize_x_ticks,
        rotation=90,
    )
    ax.text(
        i - d,
        log_spacer(train_counts[i]),
        f"{train_counts[i]:,}",
        ha="center",
        va="bottom",
        fontsize=fontsize_x_ticks,
        rotation=90,
    )
    ax.text(
        i + d,
        log_spacer(val_counts[i]),
        f"{val_counts[i]:,}",
        ha="center",
        va="bottom",
        fontsize=fontsize_x_ticks,
        rotation=90,
    )
    ax.text(
        i + d + w,
        log_spacer(test_counts[i]),
        f"{test_counts[i]:,}",
        ha="center",
        va="bottom",
        fontsize=fontsize_x_ticks,
        rotation=90,
    )

ax.legend(loc="upper right", fontsize=fontsize_legend, frameon=False)
apply_axis_fonts(ax)

fig.tight_layout()
fig.subplots_adjust(bottom=0.45)

arrow_y = -0.38
ax.annotate(
    "",
    xy=(0.97, arrow_y),
    xytext=(0.03, arrow_y),
    xycoords="axes fraction",
    textcoords="axes fraction",
    arrowprops=dict(arrowstyle="-|>", color="black", lw=1.0, mutation_scale=8),
    annotation_clip=False,
)
ax.text(
    0.5,
    arrow_y + 0.03,
    "Dataset processing",
    ha="center",
    va="bottom",
    fontsize=fontsize_x_label,
    transform=ax.transAxes,
    clip_on=False,
)

path = os.path.join(PLOT_DIR, "split_filterings.pdf")
fig.savefig(path, bbox_inches="tight")
plt.show()